In [47]:
import pandas as pd
import numpy as np
import json

In [48]:
cta_df = pd.read_parquet('../extract_ridership_data/output/cta_ridership.parquet')

In [49]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides
0,40350,UIC-Halsted,2001-01-01T00:00:00.000,U,273
1,41130,Halsted-Orange,2001-01-01T00:00:00.000,U,306
2,40760,Granville,2001-01-01T00:00:00.000,U,1059
3,40070,Jackson/Dearborn,2001-01-01T00:00:00.000,U,649
4,40090,Damen-Brown,2001-01-01T00:00:00.000,U,411
5,40590,Damen/Milwaukee,2001-01-01T00:00:00.000,U,870
6,40720,East 63rd-Cottage Grove,2001-01-01T00:00:00.000,U,391
7,41260,Austin-Lake,2001-01-01T00:00:00.000,U,399
8,40230,Cumberland,2001-01-01T00:00:00.000,U,788
9,41120,35-Bronzeville-IIT,2001-01-01T00:00:00.000,U,448


# Station features

In [50]:
cta_stations_df = pd.read_parquet('../extract_ridership_data/output/cta_stations.parquet')

In [51]:
cta_stations_df.head(10)

,stop_id,direction_id,stop_name,station_name,station_descriptive_name,map_id,ada,red,blue,g,...,p,y,pnk,o,location,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_vrxf_vc4k,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu
0,30162,W,18th (54th/Cermak-bound),18th,18th (Pink Line),40830,True,False,False,False,...,False,False,True,False,"{""latitude"":""41.857908"",""longitude"":""-87.66914...",8,14920,33,343,26
1,30161,E,18th (Loop-bound),18th,18th (Pink Line),40830,True,False,False,False,...,False,False,True,False,"{""latitude"":""41.857908"",""longitude"":""-87.66914...",8,14920,33,343,26
2,30022,N,35th/Archer (Loop-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,...,False,False,False,True,"{""latitude"":""41.829353"",""longitude"":""-87.68062...",26,14924,56,719,1
3,30023,S,35th/Archer (Midway-bound),35th/Archer,35th/Archer (Orange Line),40120,True,False,False,False,...,False,False,False,True,"{""latitude"":""41.829353"",""longitude"":""-87.68062...",26,14924,56,719,1
4,30213,N,35-Bronzeville-IIT (Harlem-bound),35th-Bronzeville-IIT,35th-Bronzeville-IIT (Green Line),41120,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",12,21194,1,25,9
5,30214,S,35-Bronzeville-IIT (63rd-bound),35th-Bronzeville-IIT,35th-Bronzeville-IIT (Green Line),41120,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",12,21194,1,25,9
6,30245,N,43rd (Harlem-bound),43rd,43rd (Green Line),41270,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.816462"",""longitude"":""-87.61902...",12,4301,4,162,9
7,30246,S,43rd (63rd-bound),43rd,43rd (Green Line),41270,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.816462"",""longitude"":""-87.61902...",12,4301,4,162,9
8,30210,S,47th (63rd-bound),47th,47th (Green Line),41080,True,False,False,True,...,False,False,False,False,"{""latitude"":""41.809209"",""longitude"":""-87.61882...",12,21192,4,448,9
9,30237,N,47th (Howard-bound),47th,47th (Red Line),41230,True,True,False,False,...,False,False,False,False,"{""latitude"":""41.810318"",""longitude"":""-87.63094...",12,14924,3,189,9


In [52]:
stop_id_unique = set(cta_stations_df["map_id"])

mask_station_id = cta_df["station_id"].isin(stop_id_unique)
# True if everything is valid
all_stations_in_stations_df = mask_station_id.all()
print(all_stations_in_stations_df)

False


In [53]:
invalid_rows = cta_df[~mask_station_id]
print(invalid_rows['stationname'].value_counts())

stationname
Randolph/Wabash     6607
Madison/Wabash      6216
Washington/State    2953
Homan                 31
Name: count, dtype: int64


All of these are closed stations; see [here](https://www.chicago-l.org/stations/randolph-wabash.html) for Randolph/Wabash, 
[here](https://www.chicago-l.org/stations/madison-wabash.html) for Madison/Wabash, and [here](https://www.chicago-l.org/stations/washington-state.html) for Washington/State. That means we can safely merge the stations df onto the ridership df and have complete
station information for active stations.

In [54]:
# First, deduplicate the stations df since it seems to have one row per cardinal direction.
cta_stations_df_merge = cta_stations_df.groupby('map_id').first().reset_index()
cta_stations_df_merge = cta_stations_df_merge[['map_id', 'red', 'blue', 'g', 'brn', 'p', 'y', 'pnk', 'o', 'location']]

# Then, add features for longitude and latitude
cta_stations_df_merge['location'] = cta_stations_df_merge['location'].apply(json.loads)
cta_stations_df_merge['lat'] = cta_stations_df_merge['location'].apply(lambda d: d.get('latitude')).astype(float)
cta_stations_df_merge['lon'] = cta_stations_df_merge['location'].apply(lambda d: d.get('longitude')).astype(float)

In [55]:
cta_df = cta_df.merge(
    cta_stations_df_merge,
    how='left',
    left_on='station_id',
    right_on='map_id'
).dropna(
    subset=['map_id']
)

In [56]:
# Create categorical line feature
conditions_cta_line = [
    cta_df['red'] == 1,
    cta_df['blue'] == 1,
    cta_df['g'] == 1,
    cta_df['brn'] == 1,
    cta_df['p'] == 1,
    cta_df['y'] == 1,
    cta_df['pnk'] == 1,
    cta_df['o'] == 1
]

choices_cta_line = [
    'red',
    'blue',
    'green',
    'brown',
    'purple',
    'yellow',
    'pink',
    'orange'
]

cta_df['line'] = np.select(conditions_cta_line, choices_cta_line, default='NA')

In [57]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,p,y,pnk,o,location,lat,lon,line
0,40350,UIC-Halsted,2001-01-01T00:00:00.000,U,273,40350,False,True,False,False,False,False,False,False,"{'latitude': '41.875474', 'longitude': '-87.64...",41.875474,-87.649707,blue
1,41130,Halsted-Orange,2001-01-01T00:00:00.000,U,306,41130,False,False,False,False,False,False,False,True,"{'latitude': '41.84678', 'longitude': '-87.648...",41.846780,-87.648088,orange
2,40760,Granville,2001-01-01T00:00:00.000,U,1059,40760,True,False,False,False,False,False,False,False,"{'latitude': '41.993664', 'longitude': '-87.65...",41.993664,-87.659202,red
3,40070,Jackson/Dearborn,2001-01-01T00:00:00.000,U,649,40070,False,True,False,False,False,False,False,False,"{'latitude': '41.878183', 'longitude': '-87.62...",41.878183,-87.629296,blue
4,40090,Damen-Brown,2001-01-01T00:00:00.000,U,411,40090,False,False,False,True,False,False,False,False,"{'latitude': '41.966286', 'longitude': '-87.67...",41.966286,-87.678639,brown
5,40590,Damen/Milwaukee,2001-01-01T00:00:00.000,U,870,40590,False,True,False,False,False,False,False,False,"{'latitude': '41.909744', 'longitude': '-87.67...",41.909744,-87.677437,blue
6,40720,East 63rd-Cottage Grove,2001-01-01T00:00:00.000,U,391,40720,False,False,True,False,False,False,False,False,"{'latitude': '41.780309', 'longitude': '-87.60...",41.780309,-87.605857,green
7,41260,Austin-Lake,2001-01-01T00:00:00.000,U,399,41260,False,False,True,False,False,False,False,False,"{'latitude': '41.887293', 'longitude': '-87.77...",41.887293,-87.774135,green
8,40230,Cumberland,2001-01-01T00:00:00.000,U,788,40230,False,True,False,False,False,False,False,False,"{'latitude': '41.984246', 'longitude': '-87.83...",41.984246,-87.838028,blue
9,41120,35-Bronzeville-IIT,2001-01-01T00:00:00.000,U,448,41120,False,False,True,False,False,False,False,False,"{'latitude': '41.831677', 'longitude': '-87.62...",41.831677,-87.625826,green


In [58]:
cta_df['line'].value_counts()

line
blue      288707
red       270329
green     246540
brown     198427
pink       99208
purple     90218
orange     63133
yellow     13926
Name: count, dtype: int64

# Date features 

## Year, month, and day

In [59]:
cta_df['date'] = pd.to_datetime(cta_df['date'])

cta_df['year'] = cta_df['date'].dt.year
cta_df['month'] = cta_df['date'].dt.month
cta_df['day'] = cta_df['date'].dt.day

## Day of week

In [60]:
cta_df['day_of_week_num'] = cta_df['date'].dt.weekday
cta_df['day_of_week_name'] = cta_df['date'].dt.day_name()

cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,...,False,"{'latitude': '41.875474', 'longitude': '-87.64...",41.875474,-87.649707,blue,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,...,True,"{'latitude': '41.84678', 'longitude': '-87.648...",41.846780,-87.648088,orange,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,...,False,"{'latitude': '41.993664', 'longitude': '-87.65...",41.993664,-87.659202,red,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,...,False,"{'latitude': '41.878183', 'longitude': '-87.62...",41.878183,-87.629296,blue,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,...,False,"{'latitude': '41.966286', 'longitude': '-87.67...",41.966286,-87.678639,brown,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,...,False,"{'latitude': '41.909744', 'longitude': '-87.67...",41.909744,-87.677437,blue,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,...,False,"{'latitude': '41.780309', 'longitude': '-87.60...",41.780309,-87.605857,green,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,...,False,"{'latitude': '41.887293', 'longitude': '-87.77...",41.887293,-87.774135,green,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,...,False,"{'latitude': '41.984246', 'longitude': '-87.83...",41.984246,-87.838028,blue,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,...,False,"{'latitude': '41.831677', 'longitude': '-87.62...",41.831677,-87.625826,green,2001,1,1,0,Monday


# Save the data

In [61]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,...,False,"{'latitude': '41.875474', 'longitude': '-87.64...",41.875474,-87.649707,blue,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,...,True,"{'latitude': '41.84678', 'longitude': '-87.648...",41.846780,-87.648088,orange,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,...,False,"{'latitude': '41.993664', 'longitude': '-87.65...",41.993664,-87.659202,red,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,...,False,"{'latitude': '41.878183', 'longitude': '-87.62...",41.878183,-87.629296,blue,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,...,False,"{'latitude': '41.966286', 'longitude': '-87.67...",41.966286,-87.678639,brown,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,...,False,"{'latitude': '41.909744', 'longitude': '-87.67...",41.909744,-87.677437,blue,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,...,False,"{'latitude': '41.780309', 'longitude': '-87.60...",41.780309,-87.605857,green,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,...,False,"{'latitude': '41.887293', 'longitude': '-87.77...",41.887293,-87.774135,green,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,...,False,"{'latitude': '41.984246', 'longitude': '-87.83...",41.984246,-87.838028,blue,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,...,False,"{'latitude': '41.831677', 'longitude': '-87.62...",41.831677,-87.625826,green,2001,1,1,0,Monday


In [62]:
cta_df.tail(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
1286285,41380,Bryn Mawr,2025-09-07,U,2045,41380,True,False,False,False,...,False,"{'latitude': '41.983504', 'longitude': '-87.65...",41.983504,-87.658840,red,2025,9,7,6,Sunday
1286286,41400,Roosevelt,2025-09-07,U,5340,41400,True,False,False,False,...,False,"{'latitude': '41.867368', 'longitude': '-87.62...",41.867368,-87.627402,red,2025,9,7,6,Sunday
1286287,41410,Chicago/Milwaukee,2025-09-07,U,1785,41410,False,True,False,False,...,False,"{'latitude': '41.896075', 'longitude': '-87.65...",41.896075,-87.655214,blue,2025,9,7,6,Sunday
1286288,41420,Addison-North Main,2025-09-07,U,10067,41420,True,False,False,False,...,False,"{'latitude': '41.947428', 'longitude': '-87.65...",41.947428,-87.653626,red,2025,9,7,6,Sunday
1286289,41430,87th,2025-09-07,U,1053,41430,True,False,False,False,...,False,"{'latitude': '41.735372', 'longitude': '-87.62...",41.735372,-87.624717,red,2025,9,7,6,Sunday
1286290,41440,Addison-Brown,2025-09-07,U,754,41440,False,False,False,True,...,False,"{'latitude': '41.947028', 'longitude': '-87.67...",41.947028,-87.674642,brown,2025,9,7,6,Sunday
1286291,41450,Chicago/State,2025-09-07,U,6021,41450,True,False,False,False,...,False,"{'latitude': '41.896671', 'longitude': '-87.62...",41.896671,-87.628176,red,2025,9,7,6,Sunday
1286292,41460,Irving Park-Brown,2025-09-07,U,985,41460,False,False,False,True,...,False,"{'latitude': '41.954521', 'longitude': '-87.67...",41.954521,-87.674868,brown,2025,9,7,6,Sunday
1286293,41480,Western-Brown,2025-09-07,U,1557,41480,False,False,False,True,...,False,"{'latitude': '41.966163', 'longitude': '-87.68...",41.966163,-87.688502,brown,2025,9,7,6,Sunday
1286294,41490,Harrison,2025-09-07,U,3463,41490,True,False,False,False,...,False,"{'latitude': '41.874039', 'longitude': '-87.62...",41.874039,-87.627479,red,2025,9,7,6,Sunday


In [63]:
cta_df.to_parquet('output/cta_ridership_with_features.parquet', engine='fastparquet', index=False)